# VoxFlow speech emotion recognition
This notebook trains a small wav2vec2-based classifier for emotion recognition from voice. The demo uses synthetic audio tones first and can be swapped to RAVDESS, CREMA-D, TESS, or SAVEE later.


In [ ]:
# Install dependencies
# %pip install -q datasets transformers accelerate librosa soundfile scikit-learn

import numpy as np
import torch
from datasets import Dataset
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, TrainingArguments, Trainer

MODEL_NAME = "facebook/wav2vec2-base"
LABELS = ["neutral", "happy", "sad", "angry", "calm", "excited"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def make_tone(label: str, duration=0.8, sr=16000):
    t = np.arange(int(sr * duration)) / sr
    freq = {"neutral": 220, "happy": 330, "sad": 180, "angry": 440, "calm": 240, "excited": 520}[label]
    wave = 0.25 * np.sin(2 * np.pi * freq * t)
    return wave.astype(np.float32)

rows = []
for label in LABELS:
    for _ in range(8):
        rows.append({"audio": make_tone(label), "label": label})

raw_dataset = Dataset.from_list(rows)
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)


def prepare_example(example):
    audio = example["audio"]
    if isinstance(audio, np.ndarray):
        audio = audio.astype(np.float32)
    features = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
    return {"input_values": features["input_values"][0], "labels": LABELS.index(example["label"])}

prepared = raw_dataset.map(prepare_example, remove_columns=raw_dataset.column_names)
prepared = prepared.train_test_split(test_size=0.2, seed=42)

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label={idx: label for idx, label in enumerate(LABELS)},
    label2id={label: idx for idx, label in enumerate(LABELS)},
)
model.to(DEVICE)

training_args = TrainingArguments(
    output_dir="./ser_demo_out",
    per_device_train_batch_size=2,
    evaluation_strategy="epoch",
    learning_rate=1e-4,
    num_train_epochs=2,
    save_strategy="epoch",
    logging_steps=5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared["train"],
    eval_dataset=prepared["test"],
)

trainer.train()
print("SER fine-tuning demo completed")
